## Paso 1 — Entorno y carga de datos

Stack: **pandas** para manipulación tabular, **numpy** para operaciones vectorizadas, **matplotlib** para visualización estática.
Se suprime el backend interactivo de matplotlib para evitar conflictos en el entorno local.

# 02 — Data Cleaning & EDA
## Solvix E-commerce Analytics

Notebook de limpieza y exploración inicial sobre dos fuentes de datos operativas de Solvix:
- **Órdenes de venta** (Nov 2025 – Abr 2026): transacciones, productos, canales y destinos
- **Meta Ads** (mismo período): inversión diaria y compras atribuidas por campaña

---

**Alcance de este notebook:**

| Paso | Acción | Decisión tomada |
|---|---|---|
| 1 | Carga e inspección inicial | Confirmar estructura y dimensiones esperadas |
| 2 | Auditoría de tipos y nulos | Detectar conversiones necesarias antes de calcular |
| 3 | Verificación de duplicados | Validar integridad del ID de orden |
| 4 | Conversión de fechas + columnas temporales | Habilitar análisis por mes, semana y día |
| 5 | Validación de rangos y categorías | Descartar valores imposibles o errores de carga |
| 6 | Exportación a  | Base limpia para SQL y Power BI |

In [5]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 5)
plt.style.use('ggplot')

print("✓ Librerías cargadas correctamente")

✓ Librerías cargadas correctamente


### Fuentes de datos

Ambos archivos provienen de exportaciones directas del sistema operativo (simulando Shopify + Meta Ads Manager).
Se leen desde  — directorio que se mantiene intacto como respaldo del dato original.

In [6]:
df_ordenes = pd.read_csv('../data/raw/solvix_ordenes_raw.csv')
df_ads     = pd.read_csv('../data/raw/solvix_meta_ads_raw.csv')

print(f"Órdenes:  {df_ordenes.shape[0]:,} filas  x  {df_ordenes.shape[1]} columnas")
print(f"Meta Ads: {df_ads.shape[0]:,} filas  x  {df_ads.shape[1]} columnas")

Órdenes:  3,500 filas  x  14 columnas
Meta Ads: 360 filas  x  5 columnas


### Inspección visual inicial

Revisión rápida de las primeras filas para confirmar estructura de columnas, separador correcto y ausencia de filas de encabezado duplicadas.

In [7]:
print("=== ÓRDENES - Primeras 3 filas ===")
df_ordenes.head(3)

=== ÓRDENES - Primeras 3 filas ===


,ID_Orden,Fecha,ID_Cliente,ID_Producto,Nombre_Producto,Categoria,Cantidad,Ingreso_Total,COGS_Total,Costo_Envio,Ganancia_Bruta,Ciudad_Destino,Fuente_Adquisicion,Estado_Envio
0,ORD-1000,2026-04-13 03:48:00,CLI-5199,P002,Mini Bicicleta Premium,Fitness,1,120,45,10.58,64.42,Bucaramanga,Campaña_Bici_Carrusel,Entregado
1,ORD-1001,2025-11-27 23:05:00,CLI-5029,P001,Vaso Térmico Inteligente Auto,Auto,1,39,15,5.97,18.03,Bogotá,Orgánico_Instagram,Entregado
2,ORD-1002,2025-11-09 01:01:00,CLI-5001,P001,Vaso Térmico Inteligente Auto,Auto,1,39,15,4.03,19.97,Bucaramanga,Orgánico_Instagram,Entregado


In [8]:
print("=== META ADS - Primeras 3 filas ===")
df_ads.head(3)

=== META ADS - Primeras 3 filas ===


,Fecha,Nombre_Campaña,Gasto_Diario_USD,Clics,Compras_Atribuidas
0,2025-11-01,Campaña_Vaso_Vertical,22.08,273,6
1,2025-11-01,Campaña_Bici_Carrusel,31.94,100,2
2,2025-11-02,Campaña_Vaso_Vertical,15.54,275,3


In [9]:
# .isnull() devuelve True/False por cada celda
# .sum() cuenta cuántos True hay por columna
print("=== NULOS EN ÓRDENES ===")
nulos_ordenes = df_ordenes.isnull().sum()
print(nulos_ordenes)
print(f"\nTotal nulos: {nulos_ordenes.sum()}")

print("\n=== NULOS EN META ADS ===")
nulos_ads = df_ads.isnull().sum()
print(nulos_ads)
print(f"\nTotal nulos: {nulos_ads.sum()}")

=== NULOS EN ÓRDENES ===
ID_Orden              0
Fecha                 0
ID_Cliente            0
ID_Producto           0
Nombre_Producto       0
Categoria             0
Cantidad              0
Ingreso_Total         0
COGS_Total            0
Costo_Envio           0
Ganancia_Bruta        0
Ciudad_Destino        0
Fuente_Adquisicion    0
Estado_Envio          0
dtype: int64

Total nulos: 0

=== NULOS EN META ADS ===
Fecha                 0
Nombre_Campaña        0
Gasto_Diario_USD      0
Clics                 0
Compras_Atribuidas    0
dtype: int64

Total nulos: 0


## Paso 2 — Auditoría de tipos y estructura

**Hallazgo clave:** la columna  carga como  en ambos DataFrames — comportamiento estándar de pandas al leer CSV.
Requiere conversión explícita a  antes de cualquier operación temporal.

El resto de columnas numéricas (, ) cargan con el tipo correcto; no se requieren casteos adicionales.

In [5]:
print("=== ÓRDENES — Estructura y tipos ===")
df_ordenes.info()

=== ÓRDENES — Estructura y tipos ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3500 entries, 0 to 3499
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ID_Orden            3500 non-null   object 
 1   Fecha               3500 non-null   object 
 2   ID_Cliente          3500 non-null   object 
 3   ID_Producto         3500 non-null   object 
 4   Nombre_Producto     3500 non-null   object 
 5   Categoria           3500 non-null   object 
 6   Cantidad            3500 non-null   int64  
 7   Ingreso_Total       3500 non-null   int64  
 8   COGS_Total          3500 non-null   int64  
 9   Costo_Envio         3500 non-null   float64
 10  Ganancia_Bruta      3500 non-null   float64
 11  Ciudad_Destino      3500 non-null   object 
 12  Fuente_Adquisicion  3500 non-null   object 
 13  Estado_Envio        3500 non-null   object 
dtypes: float64(2), int64(3), object(9)
memory usage: 382.9+ KB


In [6]:
print("=== META ADS — Estructura y tipos ===")
df_ads.info()

=== META ADS — Estructura y tipos ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 360 entries, 0 to 359
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Fecha               360 non-null    object 
 1   Nombre_Campaña      360 non-null    object 
 2   Gasto_Diario_USD    360 non-null    float64
 3   Clics               360 non-null    int64  
 4   Compras_Atribuidas  360 non-null    int64  
dtypes: float64(1), int64(2), object(2)
memory usage: 14.2+ KB


## Paso 3 — Calidad de datos: nulos y duplicados

Verificación estándar de integridad antes de transformar: ausencia de nulos confirma que la exportación fue completa.
La unicidad de  descarta dobles registros por error de sistema o exportación.

In [7]:
# .isnull() devuelve True/False por cada celda
# .sum() cuenta cuántos True hay por columna
print("=== NULOS EN ÓRDENES ===")
nulos_ordenes = df_ordenes.isnull().sum()
print(nulos_ordenes)
print(f"\nTotal nulos: {nulos_ordenes.sum()}")

print("\n=== NULOS EN META ADS ===")
nulos_ads = df_ads.isnull().sum()
print(nulos_ads)
print(f"\nTotal nulos: {nulos_ads.sum()}")

=== NULOS EN ÓRDENES ===
ID_Orden              0
Fecha                 0
ID_Cliente            0
ID_Producto           0
Nombre_Producto       0
Categoria             0
Cantidad              0
Ingreso_Total         0
COGS_Total            0
Costo_Envio           0
Ganancia_Bruta        0
Ciudad_Destino        0
Fuente_Adquisicion    0
Estado_Envio          0
dtype: int64

Total nulos: 0

=== NULOS EN META ADS ===
Fecha                 0
Nombre_Campaña        0
Gasto_Diario_USD      0
Clics                 0
Compras_Atribuidas    0
dtype: int64

Total nulos: 0


In [8]:
# .duplicated() marca True en filas que ya aparecieron antes
# .sum() cuenta cuántas hay
print("=== DUPLICADOS ===")
dup_ordenes = df_ordenes.duplicated().sum()
dup_ads     = df_ads.duplicated().sum()

print(f"Filas duplicadas en Órdenes:  {dup_ordenes}")
print(f"Filas duplicadas en Meta Ads: {dup_ads}")

# También verificamos que los ID de orden sean únicos
ids_unicos = df_ordenes['ID_Orden'].nunique()
print(f"\nIDs de orden únicos: {ids_unicos} de {len(df_ordenes)} total")
print("→ ¿Hay IDs repetidos?", "SÍ ⚠️" if ids_unicos < len(df_ordenes) else "NO ✓")

=== DUPLICADOS ===
Filas duplicadas en Órdenes:  0
Filas duplicadas en Meta Ads: 0

IDs de orden únicos: 3500 de 3500 total
→ ¿Hay IDs repetidos? NO ✓


## Paso 4 — Conversión de tipos y columnas derivadas

Conversión de  a  en ambas tablas. A partir de ahí se derivan columnas de granularidad temporal
que simplifican los agrupamientos en SQL y Power BI:

-  /  — agregaciones mensuales
-  — análisis de patrones por día
-  — tendencias semanales

In [9]:
print("Tipo ANTES de convertir:", df_ordenes['Fecha'].dtype)

# pd.to_datetime() convierte texto → fecha
df_ordenes['Fecha'] = pd.to_datetime(df_ordenes['Fecha'])
df_ads['Fecha']     = pd.to_datetime(df_ads['Fecha'])

print("Tipo DESPUÉS de convertir:", df_ordenes['Fecha'].dtype)

# Ahora podemos extraer componentes útiles
df_ordenes['Mes']          = df_ordenes['Fecha'].dt.month
df_ordenes['Nombre_Mes']   = df_ordenes['Fecha'].dt.strftime('%B')   # Ej: "November"
df_ordenes['Dia_Semana']   = df_ordenes['Fecha'].dt.day_name()        # Ej: "Monday"
df_ordenes['Semana_Anio']  = df_ordenes['Fecha'].dt.isocalendar().week.astype(int)

print("\n✓ Columnas de tiempo agregadas: Mes, Nombre_Mes, Dia_Semana, Semana_Anio")
print("\nEjemplo de las nuevas columnas:")
df_ordenes[['Fecha','Mes','Nombre_Mes','Dia_Semana','Semana_Anio']].head(4)

Tipo ANTES de convertir: object
Tipo DESPUÉS de convertir: datetime64[ns]

✓ Columnas de tiempo agregadas: Mes, Nombre_Mes, Dia_Semana, Semana_Anio

Ejemplo de las nuevas columnas:


## Paso 5 — Validación de rangos y consistencia

Criterios de validación definidos en función del negocio:
- Rango de fechas: Nov 2025 – Abr 2026 (6 meses de operación)
- Métricas numéricas: sin valores negativos ni ceros en  / 
- Categorías: valores controlados en ,  y  (sin errores de tipeo)

In [10]:
# Resumen estadístico de columnas numéricas
print("=== ESTADÍSTICAS ÓRDENES ===")
df_ordenes[['Cantidad','Ingreso_Total','COGS_Total','Costo_Envio','Ganancia_Bruta']].describe().round(2)

=== ESTADÍSTICAS ÓRDENES ===


In [11]:
# Verificar rango de fechas
print("=== RANGO DE FECHAS ===")
print(f"Fecha más antigua: {df_ordenes['Fecha'].min()}")
print(f"Fecha más reciente: {df_ordenes['Fecha'].max()}")
print(f"Período esperado:  Nov 2025 → Abr 2026")

# Verificar valores categóricos (buscar errores de tipeo)
print("\n=== VALORES ÚNICOS — Estado_Envio ===")
print(df_ordenes['Estado_Envio'].value_counts())

print("\n=== VALORES ÚNICOS — Ciudad_Destino ===")
print(df_ordenes['Ciudad_Destino'].value_counts())

print("\n=== VALORES ÚNICOS — Fuente_Adquisicion ===")
print(df_ordenes['Fuente_Adquisicion'].value_counts())

=== RANGO DE FECHAS ===
Fecha más antigua: 2025-11-01 01:17:00
Fecha más reciente: 2026-04-30 23:28:00
Período esperado:  Nov 2025 → Abr 2026

=== VALORES ÚNICOS — Estado_Envio ===
Estado_Envio
Entregado      2893
En tránsito     387
Devuelto        220
Name: count, dtype: int64

=== VALORES ÚNICOS — Ciudad_Destino ===
Ciudad_Destino
Bogotá          1412
Medellín         903
Cali             495
Barranquilla     346
Bucaramanga      344
Name: count, dtype: int64

=== VALORES ÚNICOS — Fuente_Adquisicion ===
Fuente_Adquisicion
Campaña_Vaso_Vertical    1894
Campaña_Bici_Carrusel     704
Orgánico_Instagram        543
Directo                   359
Name: count, dtype: int64


## Paso 6 — Exportación a capa procesada

Separación estricta entre datos crudos () y datos listos para análisis ().
El raw no se modifica; cualquier re-procesamiento parte desde esa fuente.

Los archivos exportados son la entrada directa para:
- **Notebook 03** — queries SQL de métricas de negocio
- **Notebook 04** — segmentación RFM de clientes
- **Dashboard Power BI** — visualización ejecutiva

In [12]:
import os
os.makedirs('../data/processed', exist_ok=True)

df_ordenes.to_csv('../data/processed/solvix_ordenes_clean.csv', index=False)
df_ads.to_csv('../data/processed/solvix_ads_clean.csv',         index=False)

print("✓ Archivos exportados a data/processed/")
print(f"  → solvix_ordenes_clean.csv  ({len(df_ordenes):,} filas, {len(df_ordenes.columns)} columnas)")
print(f"  → solvix_ads_clean.csv      ({len(df_ads):,} filas, {len(df_ads.columns)} columnas)")
print(f"\nColumnas finales en órdenes:")
print(list(df_ordenes.columns))

✓ Archivos exportados a data/processed/
  → solvix_ordenes_clean.csv  (3,500 filas, 18 columnas)
  → solvix_ads_clean.csv      (360 filas, 5 columnas)

Columnas finales en órdenes:
['ID_Orden', 'Fecha', 'ID_Cliente', 'ID_Producto', 'Nombre_Producto', 'Categoria', 'Cantidad', 'Ingreso_Total', 'COGS_Total', 'Costo_Envio', 'Ganancia_Bruta', 'Ciudad_Destino', 'Fuente_Adquisicion', 'Estado_Envio', 'Mes', 'Nombre_Mes', 'Dia_Semana', 'Semana_Anio']
